单元格 1
配体参数化：给生物素（BTN）生成 FoldX 力场参数

In [ ]:
# Parameterizing ligand: BTN (biotin) for YEATS2-BTN complex

单元格 2
环境准备

In [ ]:
from pyfoldx.handlers.fileHandler import FileHandler
from pyfoldx.structure import Structure
from pyfoldx.paramx import parameterize
import os, shutil

# ===== 按你的实际路径修改 =====
PDB_FILE   = "yeats2_BTN.pdb"      # 复合物PDB，放在notebook同目录
ROTABASE      = "/home/hkl/foldx5/rotabase.txt"         # FoldX自带的rotabase.txt（改成你的路径）
FOLDX_MOL_DIR = "/home/hkl/foldx5/molecules"       # FoldX的molecules文件夹（WSL侧）

# parameterize() 需要在当前目录读到 rotabase.txt
if not os.path.exists("rotabase.txt"):
    shutil.copy(ROTABASE, "rotabase.txt")
    print("✅ 已复制 rotabase.txt 到当前目录")
os.makedirs("paramx_output", exist_ok=True)

单元格 3
加载复合物，定位 BTN 残基

In [ ]:
structure = Structure("complex", PDB_FILE)

lig_residue = None
for chain in structure.data:
    for res in structure.data[chain]:
        if structure.data[chain][res].code == "BTN":
            lig_residue = structure.data[chain][res]
            break
    if lig_residue:
        break

if not lig_residue:
    raise ValueError("❌ 在 %s 中没找到 BTN 残基，检查HETATM里的残基名" % PDB_FILE)
lig_residue

单元格 4
提取 BTN 的坐标

In [ ]:
lig_lines = []
for chain in structure.data:
    for res in structure.data[chain]:
        if structure.data[chain][res].code == "BTN":
            lig_lines.extend(structure.data[chain][res].toPdb().split("\n"))

lig_lines = [l for l in lig_lines if l.strip()]
FileHandler.writeLines("paramx_output/ligand_full.pdb", lig_lines)
print(f"✅ 提取了 {len(lig_lines)} 行 (BTN)")
lig_lines

单元格 5
原子类型映射表（核心）

In [ ]:
# 键 = BTN 原子名；值 = 模板(原子, 残基)，参数从 rotabase 中该原子的条目复制
atomMappingsDict = {
    # —— 羧基 -COOH ——
    "C11": ("CG",  "ASP"),   # 羧基碳
    "O11": ("OD1", "ASP"),   # 羧基羰基氧（受体）
    "O12": ("OG",  "SER"),   # 羧基羟基氧（给体，带H）
    # —— 戊酸侧链 4×CH2 ——
    "C10": "C_single_link",  # → THR CG2
    "C9":  "C_single_link",
    "C8":  "C_single_link",
    "C7":  "C_single_link",
    # —— 噻吩环 ——
    "C2":  "C_ring_not_arom",# → PRO CG（环上CH）
    "C6":  "C_ring_not_arom",# 环上CH2
    "S1":  ("SD",  "MET"),   # 硫醚硫 ← 关键，模板库里唯一的硫
    # —— 脲环 ——
    "N1":  ("ND2", "ASN"),   # 脲氮（给体，带H）
    "N2":  ("ND2", "ASN"),
    "C3":  ("CG",  "ASN"),   # 脲羰基碳
    "O3":  ("OD1", "ASN"),   # 脲羰基氧（受体）
    "C4":  "C_ring_not_arom",# 环交界处CH
    "C5":  "C_ring_not_arom",# 环交界处CH
}

单元格 6
修复缺失的羧基氧 O11

In [ ]:
import numpy as np

# 先清掉上次可能写入的错误 O11 行（保证可重复运行）
lig_lines = [l for l in lig_lines if not (l.startswith(("ATOM","HETATM")) and l[12:16].strip()=="O11")]

def coord(atomname):
    for l in lig_lines:
        if l.startswith(("ATOM","HETATM")) and l[12:16].strip()==atomname:
            return np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])]), l
    raise ValueError(atomname + " not found")

(c11, l_c11), (o12, _), (c10, _) = coord("C11"), coord("O12"), coord("C10")

v1 = (o12 - c11) / np.linalg.norm(o12 - c11)
v2 = (c10 - c11) / np.linalg.norm(c10 - c11)
d  = -(v1 + v2); d = d / np.linalg.norm(d)
o11 = c11 + 1.23 * d

assert all(len("%8.3f" % v) == 8 for v in o11), "坐标超出PDB格式范围，检查结构"

serial = max(int(l[6:11]) for l in lig_lines if l.startswith(("ATOM","HETATM"))) + 1

# 按PDB标准列位逐段填充（1-based列）：atom 13-16, resName 18-20, chain 22, resSeq 23-26, xyz 31-54
L = list(" " * 80)
L[0:6]   = "HETATM"
L[6:11]  = "%5d" % serial
L[12:16] = "%-4s" % "O11"
L[17:20] = "BTN"
L[21]    = l_c11[21]     # 链ID
L[22:26] = l_c11[22:26]  # 残基序号
L[30:38] = "%8.3f" % o11[0]
L[38:46] = "%8.3f" % o11[1]
L[46:54] = "%8.3f" % o11[2]
L[54:60] = "  1.00"
L[60:66] = " 20.00"
L[76:78] = " O"
new_line = "".join(L)

lig_lines.append(new_line)
print("新增原子：\n" + new_line)

with open(PDB_FILE) as f:
    pdb_all = f.read().split("\n")
fixed = []
for l in pdb_all:
    fixed.append(l)
    if l.startswith("HETATM") and l[12:16].strip()=="O12" and l[17:20]=="BTN":
        fixed.append(new_line)
assert len(fixed) > len(pdb_all), "没找到 BTN 的 O12 行"
with open("yeats2_BTN_fixed.pdb","w") as f:
    f.write("\n".join(fixed))
print("✅ 已保存 yeats2_BTN_fixed.pdb")

单元格 7
映射完整性检查

In [ ]:
heavy = [l[12:16].strip() for l in lig_lines if l.startswith(("HETATM","ATOM")) and l[76:78].strip() != "H"]
missing = [a for a in heavy if a not in atomMappingsDict]
extra   = [k for k in atomMappingsDict if k not in heavy]
print("BTN 重原子(%d):" % len(heavy), heavy)
print("缺映射:", missing if missing else "无 ✅")
print("多余键:", extra if extra else "无 ✅")
assert not missing, "有原子没配类型，按上面列表补进字典"

单元格 8
执行参数化

In [ ]:
newMol = parameterize(lig_lines, atomMappingsDict)
assert newMol is not None, "参数化失败，看上方输出定位是哪个原子"

单元格 9
保存参数文件并部署

In [ ]:
FileHandler.writeLine("paramx_output/BTN.json", newMol.toJson())
print("✅ 已生成 paramx_output/BTN.json")

# 复制到 WSL 的 FoldX molecules 目录
if os.path.isdir(FOLDX_MOL_DIR):
    shutil.copy("paramx_output/BTN.json", os.path.join(FOLDX_MOL_DIR, "BTN.json"))
    print("✅ 已部署到", FOLDX_MOL_DIR)

# 如果还要让 Windows 上 YASARA 的 FoldX 认识 BTN：
# cp paramx_output/BTN.json "/mnt/d/YASARA/foldx_Windows (1)/molecules/"